In [1]:
# 1. 一次性安装所有需要的库，指定高版本 peft 和 transformers
!pip install -U peft>=0.18.0 transformers>=4.40.0 accelerate datasets evaluate jiwer librosa soundfile -q

# 2. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
from datasets import load_from_disk
import evaluate
import numpy as np
import re

# ====================== 1. 路径设置（Colab） ======================
DATASET_PATH = "/content/drive/MyDrive/COMP4330_Whisper_MCE/mce_hf_dataset"
MODEL_PATH   = "/content/drive/MyDrive/COMP4330_Whisper_MCE/whisper-small-mce-lora-final"



# ====================== 2. 加载数据 ======================
dataset = load_from_disk(DATASET_PATH)
test_dataset = dataset["test"]
print(f"Test set 笔数：{len(test_dataset)}")

# ====================== 3. 加载 Fine-tuned 模型 ======================
print("加载模型中...")
base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model = model.merge_and_unload()          # 合并 LoRA 权重
model = model.to("cuda")
model.eval()

processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="zh", task="transcribe")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# ====================== 4. 工具函数 ======================
def get_audio_array(audio_item):
    if hasattr(audio_item, "get_all_samples"):
        samples = audio_item.get_all_samples()
        array = samples.data.numpy().squeeze()
        sr = samples.sample_rate
    elif isinstance(audio_item, dict) and "array" in audio_item:
        array = audio_item["array"]
        sr = audio_item["sampling_rate"]
    else:
        array = np.array(audio_item)
        sr = 16000
    return array, sr

def clean_text(text):
    text = str(text).strip()
    text = re.sub(r'^[\"\'「」『』]+|[\"\'「」『』]+$', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def map_to_pred(batch):
    array, sr = get_audio_array(batch["audio"])
    if array.ndim > 1:
        array = array.mean(axis=0)
    array = array.astype(np.float32)

    input_features = processor(
        array,
        sampling_rate=sr,
        return_tensors="pt"
    ).input_features.to("cuda")

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            language="zh",
            task="transcribe",
            max_length=225
        )

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    batch["prediction"] = transcription
    return batch

# ====================== 5. 先预览 5 笔 ======================
print("先预览 5 笔结果...")
small_test = test_dataset.select(range(5))
result_small = small_test.map(map_to_pred, remove_columns=["audio"])

for i in range(5):
    print(f"【参考】{clean_text(result_small[i]['sentence'])}")
    print(f"【预测】{clean_text(result_small[i]['prediction'])}")
    print("-" * 60)

# ====================== 6. 完整评估 ======================
print("\n开始完整评估 Fine-tuned 模型（约需 15~20 分钟）...")
result = test_dataset.map(map_to_pred, remove_columns=["audio"])

predictions = [clean_text(x["prediction"]) for x in result]
references  = [clean_text(x["sentence"]) for x in result]

cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

cer = cer_metric.compute(predictions=predictions, references=references)
wer = wer_metric.compute(predictions=predictions, references=references)

print(f"\n========== Fine-tuned 结果 ==========")
print(f"Fine-tuned Whisper-small (LoRA) CER：{cer * 100:.2f}%")
print(f"Fine-tuned Whisper-small (LoRA) WER：{wer * 100:.2f}%")

print(f"\n========== 与 Baseline 比较 ==========")
print(f"Baseline CER：        47.41%")
print(f"Fine-tuned CER：      {cer * 100:.2f}%")
print(f"绝对下降：            {47.41 - cer * 100:.2f} 百分点")
print(f"相对错误率下降：      {(47.41 - cer * 100) / 47.41 * 100:.1f}%")

Test set 笔数：1405
加载模型中...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

先预览 5 笔结果...


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

【参考】我有個好特別嘅愛好就係集郵，雖然聽落去有啲老土，但當你開始揀返啲stamp同了解背後嘅story，你就會覺得有好大嘅滿足感。
【预测】我有個好特別嘅愛好就係雜遊，雖然聽落去有啲老土，但當你開始返啲stamps同了解背後嘅story，你就會覺得好大嘅滿足感。
------------------------------------------------------------
【参考】Eat more fruit and vegetables: 佢哋豐富嘅nutrients同流食纖維, 可以增強我哋嘅immune system同防止疾病.
【预测】Eat my fruit and vegetables，你嘅腹肝 nutrients 同流血纪率，可以增強我哋嘅 immune system 同防止疾病。
------------------------------------------------------------
【参考】漢堡和薯條係我最好嘅grab-and-go food！
【预测】漢堡和食條係我最好嘅grab and go food.
------------------------------------------------------------
【参考】Stay focused, 保持专注，才能够把工作做得更加完成。
【预测】Stay focused, 保持專注，才能夠把工作做得更加完成。
------------------------------------------------------------
【参考】學習可以係fun嘅，唔係悶嘅。將每樣事情變成game，喺其中find joy吧！
【预测】學習可以係fun嘅，唔係悶嘅，將每樣事情變成game，喺其中返joy吧。
------------------------------------------------------------

开始完整评估 Fine-tuned 模型（约需 15~20 分钟）...


Map:   0%|          | 0/1405 [00:00<?, ? examples/s]


========== Fine-tuned 结果 ==========
Fine-tuned Whisper-small (LoRA) CER：17.84%
Fine-tuned Whisper-small (LoRA) WER：80.07%

========== 与 Baseline 比较 ==========
Baseline CER：        47.41%
Fine-tuned CER：      17.84%
绝对下降：            29.57 百分点
相对错误率下降：      62.4%
